In [2]:
# Import Libraries
import pandas as pd
import numpy as np
import re

In [4]:
# load the dataset
df = pd.read_csv("/content/sample_data/project_risk_raw_dataset.csv")
print("Shape of Dataset: ",df.shape)

Shape of Dataset:  (4000, 51)


In [5]:
df.head()

,Project_ID,Project_Type,Team_Size,Project_Budget_USD,Estimated_Timeline_Months,Complexity_Score,Stakeholder_Count,Methodology_Used,Team_Experience_Level,Past_Similar_Projects,...,Industry_Volatility,Client_Experience_Level,Change_Control_Maturity,Risk_Management_Maturity,Team_Colocation,Documentation_Quality,Project_Start_Month,Current_Phase_Duration_Months,Seasonal_Risk_Factor,Risk_Level
0,PROJ_0001,Construction,32,1526276.55,32,9.70,16,Waterfall,Senior,3,...,Extreme,First-time,Basic,Basic,Fully Colocated,Good,10,5,1.0,High
1,PROJ_0002,Manufacturing,2,390790.15,9,2.72,9,Kanban,Mixed,0,...,Stable,Occasional,Advanced,Formal,Fully Remote,Poor,9,3,1.0,Low
2,PROJ_0003,Manufacturing,2,246674.76,6,2.04,7,Agile,Mixed,1,...,Stable,Regular,NaN,NaN,Hybrid,Good,5,1,1.0,Medium
3,PROJ_0004,IT,12,1427830.63,17,7.54,16,Scrum,Mixed,0,...,Extreme,Strategic,Formal,Basic,Hybrid,Basic,12,6,1.1,High
4,PROJ_0005,Construction,24,1696746.64,24,6.68,17,Hybrid,Junior,0,...,Moderate,Occasional,Basic,NaN,Partially Colocated,Basic,9,6,1.0,High


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 51 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Project_ID                       4000 non-null   object 
 1   Project_Type                     4000 non-null   object 
 2   Team_Size                        4000 non-null   int64  
 3   Project_Budget_USD               4000 non-null   float64
 4   Estimated_Timeline_Months        4000 non-null   int64  
 5   Complexity_Score                 4000 non-null   float64
 6   Stakeholder_Count                4000 non-null   int64  
 7   Methodology_Used                 4000 non-null   object 
 8   Team_Experience_Level            4000 non-null   object 
 9   Past_Similar_Projects            4000 non-null   int64  
 10  External_Dependencies_Count      4000 non-null   int64  
 11  Change_Request_Frequency         4000 non-null   float64
 12  Project_Phase       

In [7]:
df.nunique()

,0
Project_ID,4000
Project_Type,6
Team_Size,49
Project_Budget_USD,4000
Estimated_Timeline_Months,35
Complexity_Score,796
Stakeholder_Count,27
Methodology_Used,5
Team_Experience_Level,4
Past_Similar_Projects,11


In [8]:
# 1. Convert Pascal_Case to snake_case
# This ensures AlloyDB doesn't require double quotes for queries
def to_snake_case(name):
    # Handles transitions like ProjectID to project_id or Project_Type to project_type
    s1 = re.sub('(.)([A-Z][a-z]+)', r'\1_\2', name)
    return re.sub('([a-z0-9])([A-Z])', r'\1_\2', s1).lower().replace('__', '_')

In [9]:
df.columns = [to_snake_case(col) for col in df.columns]

In [10]:
df.columns

Index(['project_id', 'project_type', 'team_size', 'project_budget_usd',
       'estimated_timeline_months', 'complexity_score', 'stakeholder_count',
       'methodology_used', 'team_experience_level', 'past_similar_projects',
       'external_dependencies_count', 'change_request_frequency',
       'project_phase', 'requirement_stability', 'team_turnover_rate',
       'vendor_reliability_score', 'historical_risk_incidents',
       'communication_frequency', 'regulatory_compliance_level',
       'technology_familiarity', 'geographical_distribution',
       'stakeholder_engagement_level', 'schedule_pressure',
       'budget_utilization_rate', 'executive_sponsorship', 'funding_source',
       'market_volatility', 'integration_complexity', 'resource_availability',
       'priority_level', 'organizational_change_frequency',
       'cross_functional_dependencies', 'previous_delivery_success_rate',
       'technical_debt_level', 'project_manager_experience',
       'org_process_maturity', 'dat

In [11]:
# 2. Advanced Missing Value Imputation
# For the categorical columns with high NaN counts:
missing_cat_cols = ['tech_environment_stability', 'change_control_maturity', 'risk_management_maturity']
for col in missing_cat_cols:
    # Fill with 'Not Assessed' so the AI knows it's an unknown factor, not a system error
    df[col] = df[col].fillna('Not Assessed')

In [12]:
# 3. Handle Numerical Outliers
# Ensure budget and scores are positive and realistic
# (Standard practice for a Data Science grad to show data integrity)
df = df[df['project_budget_usd'] > 0]
df = df[df['complexity_score'] >= 0]

In [13]:
# 4. Feature Scaling/Standardization (Optional but looks great in PPT)
# Create a 'Risk Score' numerical mapping for the 'risk_level' column
risk_map = {'Low': 1, 'Medium': 2, 'High': 3, 'Extreme': 4}
df['risk_numeric_score'] = df['risk_level'].map(risk_map).fillna(0)

In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 52 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   project_id                       4000 non-null   object 
 1   project_type                     4000 non-null   object 
 2   team_size                        4000 non-null   int64  
 3   project_budget_usd               4000 non-null   float64
 4   estimated_timeline_months        4000 non-null   int64  
 5   complexity_score                 4000 non-null   float64
 6   stakeholder_count                4000 non-null   int64  
 7   methodology_used                 4000 non-null   object 
 8   team_experience_level            4000 non-null   object 
 9   past_similar_projects            4000 non-null   int64  
 10  external_dependencies_count      4000 non-null   int64  
 11  change_request_frequency         4000 non-null   float64
 12  project_phase       

In [15]:
# 5. Export for Cloud Storage
df.to_csv('cleaned_project_risk_data.csv', index=False)
print("Data fully sanitized and exported.")

Data fully sanitized and exported.
